# CPU K-Means: dimension sweep

We benchmark float64 K-Means at D=2, 4 and 9 using the same rows, K and projected canonical initial centroids as the GPU notebooks.

Run the configuration and data checks before the batch. Setup and result interpretation are in [readme.md](readme.md) and [the design notes](docs/IMPLEMENTATION_DESIGN.md).


## 1. Runtime

Mount Drive on Colab and load the CPU dependencies.


In [ ]:
import sys, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

for pkg in ("numba", "threadpoolctl", "psutil"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Running on Colab" if IN_COLAB else "Running locally")


## 2. Configuration

`FEATURE_DIMS` selects the feature subsets. Keep `SWEEP_CASES = None` to run all discovered datasets.


In [ ]:
import os
from pathlib import Path
import numpy as np

if IN_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/kmeans_data")
    OUT_DIR = DATA_DIR / "results_cpu_baseline_v3_aligned_dimension_sweep"
else:
    DATA_DIR = Path(".")
    OUT_DIR = Path("results_cpu_baseline_v3_aligned_dimension_sweep")

SUBFOLDER_PATTERN = "rows*"
FILE_PATTERN = "synthetic_data_*_rows_*_cols_*_clusters.csv"
SIZES = None
RESUME = True

ID_COLUMN = "auto"
LABEL_COLUMN = "auto"
SAVE_ASSIGNMENTS = True
ASSIGNMENTS_DIR = (
    Path("/content/assignments_cpu_baseline_v3_aligned_dimension_sweep")
    if IN_COLAB
    else OUT_DIR / "assignments"
)
HISTORY_DIR = OUT_DIR / "history"

INIT_DIR = DATA_DIR / "results_gpu_atomic_v3"
REQUIRE_CANONICAL_INIT = True
SEED = 42

TOL = 1e-4
ASSIGNMENT_CHANGE_FRACTION_TOL = 1e-4
ASSIGNMENT_STABLE_ITERS = 3
MAX_ITER = 400

WARMUP_RUNS = 1
BENCH_REPEATS = 5
BENCH_ITERS = 20
BENCH_MIN_SECONDS = 0.5
BENCH_MAX_ITERS = 20_000

N_THREADS = os.cpu_count()
THREAD_CANDIDATES = None

DTYPE = np.float64

from pathlib import Path

# Select the first D named features; each dataset keeps all its rows.
FEATURE_DIMS = [2, 4, 9]

SWEEP_CASES = None

CANONICAL_INIT_EXTRA_FOLDERS = ["results_gpu_atomic_v3 2"]

def resolve_canonical_initial_path(primary_directory, base_tag):
    primary = Path(primary_directory)
    folders = [primary] + [primary.parent / name for name in CANONICAL_INIT_EXTRA_FOLDERS]
    filename = f"initial_centroids_{base_tag}.csv"
    candidates = [folder / filename for folder in folders]
    for path in candidates:
        if path.exists():
            return path
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Full-9D canonical initialization missing; searched: {searched}")

def select_feature_columns(available, dimension):
    if isinstance(dimension, (bool, np.bool_)) or not isinstance(dimension, (int, np.integer)):
        raise ValueError("Feature dimensions must be integers")
    if not 1 <= dimension <= 9:
        raise ValueError(f"D={dimension}: existing data supports dimensions 1 through 9")
    selected = [f"col_{i}" for i in range(1, int(dimension) + 1)]
    missing = [c for c in selected if c not in available]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    return selected

def dimension_tag(base_tag, dimension):
    return f"{base_tag}_D{int(dimension)}"

if not FEATURE_DIMS or len(set(FEATURE_DIMS)) != len(FEATURE_DIMS):
    raise ValueError("FEATURE_DIMS must be nonempty and contain no duplicates")
for _dimension in FEATURE_DIMS:
    select_feature_columns([f"col_{i}" for i in range(1, 10)], _dimension)
print("Dimension sweep:", FEATURE_DIMS, "| (N, K) filter:", SWEEP_CASES)


In [ ]:
import os, re
base = str(DATA_DIR)
print("DATA_DIR:", base)
for root, dirs, files in os.walk(base):
    depth = root[len(base):].count(os.sep)
    if depth > 3:
        continue
    print("  " * depth + (os.path.basename(root) or root) + f"/  ({len(files)} files)")
    for f in files:
        if "10000000" in f or "10m" in f.lower():
            print("  " * (depth + 1) + "->", f)

print("\nAnything named like 10M elsewhere in MyDrive:")
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for name in dirs + files:
        if "10000000" in name or re.search(r"10m", name, re.I):
            print("  ", os.path.join(root, name))


## 3. Environment

We save hardware, library versions and benchmark settings with the results.


In [ ]:
import json, platform, re, time, warnings, shutil, zipfile, hashlib
import pandas as pd
from threadpoolctl import threadpool_info, threadpool_limits

try:
    import numba
    from numba import njit, prange
    HAVE_NUMBA = True
    numba.set_num_threads(min(N_THREADS, numba.config.NUMBA_NUM_THREADS))
except ImportError:
    HAVE_NUMBA = False
IMPLEMENTATION = "numba" if HAVE_NUMBA else "numpy"
OUT_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

def cpu_info():
    info = {"cpu_model": platform.processor() or "unknown", "physical_cores": None}
    try:
        if sys.platform == "darwin":
            info["cpu_model"] = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        elif sys.platform.startswith("linux"):
            for line in open("/proc/cpuinfo"):
                if line.startswith("model name"):
                    info["cpu_model"] = line.split(":", 1)[1].strip()
                    break
        import psutil
        info["physical_cores"] = psutil.cpu_count(logical=False)
        info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        pass
    return info

ENV = {
    "feature_dimensions": FEATURE_DIMS,
    "feature_selection": "first D named col_* features",
    "sweep_cases": SWEEP_CASES,
    "runtime": "Google Colab (CPU)" if IN_COLAB else "local",
    **cpu_info(),
    "logical_cores": os.cpu_count(),
    "max_threads": N_THREADS,
    "implementation": IMPLEMENTATION,
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "numba": numba.__version__ if HAVE_NUMBA else None,
    "pandas": pd.__version__,
    "blas": [{k: d.get(k) for k in ("internal_api", "version", "num_threads")} for d in threadpool_info()],
    "settings": {"TOL": TOL, "MAX_ITER": MAX_ITER, "WARMUP_RUNS": WARMUP_RUNS, "BENCH_REPEATS": BENCH_REPEATS,
                 "BENCH_ITERS": BENCH_ITERS, "BENCH_MIN_SECONDS": BENCH_MIN_SECONDS, "SEED": SEED,
                 "canonical_init_dir": str(INIT_DIR),
                 "assignment_change_fraction_tolerance": ASSIGNMENT_CHANGE_FRACTION_TOL,
                 "assignment_stable_iterations": ASSIGNMENT_STABLE_ITERS,
                 "THREAD_CANDIDATES": THREAD_CANDIDATES or sorted({1, N_THREADS}),
                 "dtype": np.dtype(DTYPE).name, "centroid_sums": "float64"},
}
with open(OUT_DIR / "environment.json", "w") as fh:
    json.dump(ENV, fh, indent=2)
if not HAVE_NUMBA:
    warnings.warn("Numba is not available: falling back to the NumPy implementation, which is much slower.")
ENV


## 4. Data and initialization

We select `col_1,...,col_D` and the same columns from full-9D canonical centroids; ID and true labels stay outside clustering.


In [ ]:
NAME_RE = re.compile(r"synthetic_data_(\d+)_rows_(\d+)_cols_(\d+)_clusters")
LABEL_NAMES = {"true_label", "label", "labels", "cluster", "clusters", "cluster_id", "target", "class", "y"}
INDEX_NAMES = {"id", "index", "idx", "row", "row_id"}

def parse_name(path):
    m = NAME_RE.search(Path(path).name)
    if not m:
        raise ValueError(f"Unexpected file name: {path}")
    n, cols, k = map(int, m.groups())
    return {"file": m.group(0), "rows": n, "cols": cols, "K": k}

def _inside_out_dir(p):
    try:
        return OUT_DIR.resolve() in Path(p).resolve().parents
    except Exception:
        return False

def find_data_files():
    'Data files in DATA_DIR/<SUBFOLDER_PATTERN>/** (any depth), one per file name.'
    base = Path(DATA_DIR)
    if not base.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {base}")
    roots = [d for d in sorted(base.glob(SUBFOLDER_PATTERN)) if d.is_dir()] if SUBFOLDER_PATTERN else [base]
    if not roots:
        raise FileNotFoundError(f"No folders matching {SUBFOLDER_PATTERN!r} in {base}")
    found = {}
    for root in roots:
        for p in sorted(root.rglob(FILE_PATTERN)):
            if _inside_out_dir(p):
                continue
            if p.name in found:
                warnings.warn(f"Duplicate {p.name}: using {found[p.name]}, ignoring {p}")
                continue
            found[p.name] = p
    for p in found.values():
        for part in p.relative_to(base).parts[:-1]:
            m = re.fullmatch(r"rows_?(\d+)", part)
            if m and int(m.group(1)) != parse_name(p)["rows"]:
                warnings.warn(f"{p}: folder {part} does not match the row count in the file name")
    return sorted(found.values(), key=lambda p: (parse_name(p)["rows"], parse_name(p)["K"]))

def _is_number(tok):
    try:
        float(tok)
        return True
    except ValueError:
        return False

def _detect_header(path):
    'Header = some field in line 1 is text where line 2 has a number (an ID column is text on both lines).'
    with open(path) as fh:
        l1 = fh.readline().strip().split(",")
        l2 = fh.readline().strip().split(",")
    return 0 if any(not _is_number(a) and _is_number(b) for a, b in zip(l1, l2)) else None

def _looks_like_index(col, values):
    name = str(col).strip().lower()
    if name in INDEX_NAMES or name.startswith("unnamed"):
        return True
    v = values.to_numpy()
    return len(v) > 1 and np.array_equal(v, np.arange(v[0], v[0] + len(v)))

def _read_csv(path, header, **kw):

    return pd.read_csv(path, header=header, index_col=False, **kw)

def load_dataset(path, meta, feature_dim=9):
    'Returns (X float64 selected features, layout). The ID column is only recorded in `layout`.'
    header = _detect_header(path)
    sniff = _read_csv(path, header, nrows=2000)
    empty = [c for c in sniff.columns if sniff[c].isna().all()]
    cols = [c for c in sniff.columns if c not in empty]
    as_num = {c: pd.to_numeric(sniff[c], errors="coerce") for c in cols}
    numeric = [c for c in cols if as_num[c].notna().all()]
    text = [c for c in cols if c not in numeric]

    if ID_COLUMN == "auto":
        named = [c for c in cols if str(c).strip().lower() in INDEX_NAMES]
        id_col = named[0] if named else (text[0] if len(text) == 1 else None)
    else:
        id_col = ID_COLUMN
        if id_col is not None and id_col not in cols:
            raise ValueError(f"{Path(path).name}: ID column {ID_COLUMN!r} not found in {cols}")
    numeric = [c for c in numeric if c != id_col]
    index_cols = [c for c in numeric if _looks_like_index(c, as_num[c])]
    numeric = [c for c in numeric if c not in index_cols]
    skipped = [c for c in text if c != id_col] + index_cols + empty

    label = None
    if LABEL_COLUMN == "auto":
        named = [c for c in numeric if str(c).strip().lower() in LABEL_NAMES]
        if named:
            label = named[0]
        elif numeric and (as_num[numeric[-1]] == as_num[numeric[-1]].round()).all():
            label = numeric[-1]
    elif LABEL_COLUMN is not None:
        label = LABEL_COLUMN
        if label not in numeric:
            raise ValueError(f"{Path(path).name}: label column {LABEL_COLUMN!r} not found among {numeric}")
    features = [c for c in numeric if c != label]
    if not features:
        raise ValueError(f"{Path(path).name}: no numeric feature columns found")

    available_features = list(features)
    features = select_feature_columns(available_features, feature_dim)
    df = _read_csv(path, header, usecols=features, dtype={c: DTYPE for c in features})
    X = np.ascontiguousarray(df[features].to_numpy(dtype=DTYPE))
    layout = {"header": header is not None, "id_column": id_col, "feature_columns": features,
              "label_column": label, "skipped": skipped}
    n_cols = len(available_features) + (label is not None) + (id_col is not None)
    if n_cols != meta["cols"]:
        warnings.warn(f"{Path(path).name}: ID + features + label = {n_cols} columns, "
                      f"but the file name says {meta['cols']}. Check the layout: {layout}")
    return X, layout

def read_ids(path, layout, n):
    'IDs in file order (row numbers if the file has no ID column).'
    if layout["id_column"] is None:
        return pd.Series(np.arange(n), name="row")
    header = 0 if layout["header"] else None
    ids = _read_csv(path, header, usecols=[layout["id_column"]], dtype={layout["id_column"]: str})[layout["id_column"]]
    if len(ids) != n:
        raise ValueError(f"{Path(path).name}: {len(ids)} IDs but {n} feature rows")
    return ids.rename("ID")

def initial_centroid_sha256(C):
    arr = np.ascontiguousarray(
        np.asarray(C, dtype=np.float64)
    )
    return hashlib.sha256(arr.tobytes()).hexdigest()

def init_label():
    return f"canonical Atomic V3 from {INIT_DIR}"

def _read_canonical_initial_centroids(path, K, d, feature_cols):
    df = pd.read_csv(path)

    if "cluster" in df.columns:
        df = df.drop(columns=["cluster"])

    if feature_cols and all(c in df.columns for c in feature_cols):
        df = df[list(feature_cols)]
    else:
        df = df.select_dtypes(include=[np.number])

    C = df.to_numpy(dtype=np.float64)

    if C.shape != (K, d):
        raise ValueError(
            f"{path} has shape {C.shape}, expected {(K, d)}"
        )

    return np.ascontiguousarray(C, dtype=DTYPE)

def initial_centroids(meta, X, feature_cols):
    'Load the exact centroid matrix produced by Atomic V3.'
    K, d = meta["K"], X.shape[1]

    name = f"initial_centroids_{meta['file']}.csv"
    path = resolve_canonical_initial_path(INIT_DIR, meta["file"])

    if not path.exists():
        raise FileNotFoundError(
            f"Canonical initialization is missing: {path}\n"
            "Run the Atomic V3 notebook first, or copy its initial_centroids_*.csv "
            "files into results_gpu_atomic_v3."
        )

    C = _read_canonical_initial_centroids(
        path,
        K,
        d,
        feature_cols
    )

    local = pd.DataFrame(
        C,
        columns=feature_cols
    )
    local.insert(0, "cluster", np.arange(K))
    local.to_csv(
        OUT_DIR / f"initial_centroids_{dimension_tag(meta['file'], d)}.csv",
        index=False
    )

    return C, init_label()


Check the detected feature columns and ID layout before starting the full batch.


In [ ]:
_files = find_data_files()
print(f"{len(_files)} data files found under {DATA_DIR}")
for _p in _files:
    print("  ", _p.relative_to(DATA_DIR))
if _files:
    _p = _files[0]
    with open(_p) as fh:
        print("\nFirst lines of", _p.name)
        for _ in range(3):
            print("   ", fh.readline().rstrip()[:160])
    _X, _layout = load_dataset(_p, parse_name(_p), FEATURE_DIMS[0])
    print("layout:", _layout)
    print("X:", _X.shape, _X.dtype)
    del _X


## 5. Lloyd iterations

We assign points, accumulate sums and counts, update nonempty centroids, and retain the previous mean for empty clusters.


In [ ]:
def _update(C, sums, counts):
    newC = C.astype(np.float64)
    nz = counts > 0

    newC[nz] = sums[nz] / counts[nz, None]

    shift = float(
        np.sqrt(
            ((newC - C) ** 2).sum(axis=1)
        ).max()
    )

    return newC.astype(C.dtype), shift

if HAVE_NUMBA:
    @njit(parallel=True, fastmath=True)
    def _lloyd_step(X, C, labels, n_blocks):
        'One assignment/update accumulation pass.'
        n, d = X.shape
        K = C.shape[0]

        sums = np.zeros(
            (n_blocks, K, d),
            dtype=np.float64
        )
        counts = np.zeros(
            (n_blocks, K),
            dtype=np.int64
        )
        changed_blocks = np.zeros(
            n_blocks,
            dtype=np.int64
        )

        block = (
            n + n_blocks - 1
        ) // n_blocks

        for b in prange(n_blocks):
            start = b * block
            end = min(n, start + block)

            local_changed = 0

            for i in range(start, end):
                best_k = 0
                best_d = np.inf

                for k in range(K):
                    dist = np.float64(0.0)

                    for j in range(d):
                        diff = (
                            np.float64(X[i, j])
                            - np.float64(C[k, j])
                        )
                        dist += diff * diff

                    if dist < best_d:
                        best_d = dist
                        best_k = k

                if labels[i] != best_k:
                    local_changed += 1

                labels[i] = best_k
                counts[b, best_k] += 1

                for j in range(d):
                    sums[b, best_k, j] += X[i, j]

            changed_blocks[b] = local_changed

        return (
            sums.sum(axis=0),
            counts.sum(axis=0),
            changed_blocks.sum()
        )

    @njit(parallel=True, fastmath=True)
    def _assign(X, C, labels, dists):
        n, d = X.shape
        K = C.shape[0]

        for i in prange(n):
            best_k = 0
            best_d = np.inf

            for k in range(K):
                dist = 0.0

                for j in range(d):
                    diff = (
                        np.float64(X[i, j])
                        - np.float64(C[k, j])
                    )
                    dist += diff * diff

                if dist < best_d:
                    best_d = dist
                    best_k = k

            labels[i] = best_k
            dists[i] = np.sqrt(best_d)

def kmeans_numba(
    X,
    C0,
    max_iter=MAX_ITER,
    tol=TOL,
    assignment_fraction_tol=ASSIGNMENT_CHANGE_FRACTION_TOL,
    stable_iters=ASSIGNMENT_STABLE_ITERS,
    collect_history=False
):
    C = np.array(
        C0,
        dtype=X.dtype,
        copy=True
    )

    labels = np.full(
        X.shape[0],
        -1,
        dtype=np.int32
    )

    n_blocks = max(
        1,
        numba.get_num_threads() * 4
    )

    shift = np.inf
    changed = X.shape[0]
    changed_fraction = 1.0
    stable_streak = 0
    reason = "max_iterations"
    history = []

    for it in range(1, max_iter + 1):
        sums, counts, changed = _lloyd_step(
            X,
            C,
            labels,
            n_blocks
        )

        changed = int(changed)
        changed_fraction = (
            changed / float(X.shape[0])
        )

        C, shift = _update(
            C,
            sums,
            counts
        )

        centroid_met = (
            tol >= 0
            and shift < tol
        )

        assignment_met = (
            assignment_fraction_tol >= 0
            and changed_fraction
            <= assignment_fraction_tol
        )

        if assignment_met:
            stable_streak += 1
        else:
            stable_streak = 0

        stability_met = (
            assignment_fraction_tol >= 0
            and stable_streak >= stable_iters
        )

        if collect_history:
            history.append({
                "iteration": it,
                "assignment_changes": changed,
                "assignment_change_fraction": changed_fraction,
                "assignment_change_percent": changed_fraction * 100.0,
                "max_centroid_movement": shift,
                "stable_fraction_streak": stable_streak,
                "centroid_tolerance_met": centroid_met,
                "assignment_fraction_tolerance_met": assignment_met,
                "assignment_stability_met": stability_met,
            })

        if centroid_met or stability_met:
            if centroid_met and stability_met:
                reason = "both"
            elif stability_met:
                reason = "assignment_fraction_stable"
            else:
                reason = "centroid_tolerance"

            break

    return (
        C,
        labels,
        it,
        shift,
        changed,
        changed_fraction,
        stable_streak,
        reason,
        history,
    )

def kmeans_numpy(
    X,
    C0,
    max_iter=MAX_ITER,
    tol=TOL,
    assignment_fraction_tol=ASSIGNMENT_CHANGE_FRACTION_TOL,
    stable_iters=ASSIGNMENT_STABLE_ITERS,
    collect_history=False,
    chunk=262_144
):
    'NumPy fallback using the same convergence rule.'
    mu = X.mean(
        axis=0,
        dtype=np.float64
    ).astype(X.dtype)

    Xc = X - mu
    C = (
        np.array(C0, dtype=X.dtype, copy=True)
        - mu
    )

    n, d = X.shape
    K = C.shape[0]

    labels = np.full(
        n,
        -1,
        dtype=np.int64
    )

    shift = np.inf
    changed = n
    changed_fraction = 1.0
    stable_streak = 0
    reason = "max_iterations"
    history = []

    for it in range(1, max_iter + 1):
        C64 = C.astype(np.float64)
        c_sq = (C64 * C64).sum(axis=1)

        changed = 0

        for s in range(0, n, chunk):
            e = min(s + chunk, n)

            D = (
                Xc[s:e].astype(np.float64)
                @ C64.T
            )
            D *= -2
            D += c_sq

            new_labels = D.argmin(axis=1)

            changed += int(
                np.count_nonzero(
                    labels[s:e] != new_labels
                )
            )

            labels[s:e] = new_labels

        changed_fraction = (
            changed / float(n)
        )

        counts = np.bincount(
            labels,
            minlength=K
        )

        sums = np.empty(
            (K, d),
            dtype=np.float64
        )

        for j in range(d):
            sums[:, j] = np.bincount(
                labels,
                weights=Xc[:, j],
                minlength=K
            )

        C, shift = _update(
            C,
            sums,
            counts
        )

        centroid_met = (
            tol >= 0
            and shift < tol
        )

        assignment_met = (
            assignment_fraction_tol >= 0
            and changed_fraction
            <= assignment_fraction_tol
        )

        if assignment_met:
            stable_streak += 1
        else:
            stable_streak = 0

        stability_met = (
            assignment_fraction_tol >= 0
            and stable_streak >= stable_iters
        )

        if collect_history:
            history.append({
                "iteration": it,
                "assignment_changes": changed,
                "assignment_change_fraction": changed_fraction,
                "assignment_change_percent": changed_fraction * 100.0,
                "max_centroid_movement": shift,
                "stable_fraction_streak": stable_streak,
                "centroid_tolerance_met": centroid_met,
                "assignment_fraction_tolerance_met": assignment_met,
                "assignment_stability_met": stability_met,
            })

        if centroid_met or stability_met:
            if centroid_met and stability_met:
                reason = "both"
            elif stability_met:
                reason = "assignment_fraction_stable"
            else:
                reason = "centroid_tolerance"

            break

    return (
        C + mu,
        labels,
        it,
        shift,
        changed,
        changed_fraction,
        stable_streak,
        reason,
        history,
    )

def assign_to_centroids(X, C):
    'Nearest final centroid and distance for every row.'
    if HAVE_NUMBA:
        labels = np.empty(
            X.shape[0],
            dtype=np.int32
        )
        dists = np.empty(
            X.shape[0],
            dtype=np.float64
        )

        _assign(
            X,
            np.ascontiguousarray(C, dtype=X.dtype),
            labels,
            dists
        )

        return labels, dists

    labels = np.empty(
        X.shape[0],
        dtype=np.int64
    )
    dists = np.empty(
        X.shape[0],
        dtype=np.float64
    )
    C64 = C.astype(np.float64)

    for s in range(
        0,
        X.shape[0],
        262_144
    ):
        e = min(
            s + 262_144,
            X.shape[0]
        )

        D = (
            (
                X[s:e, None, :].astype(np.float64)
                - C64[None]
            ) ** 2
        ).sum(axis=2)

        labels[s:e] = D.argmin(axis=1)
        dists[s:e] = np.sqrt(
            D.min(axis=1)
        )

    return labels, dists

if HAVE_NUMBA:
    @njit(parallel=True, fastmath=True)
    def _sqdist_to(X, c, out):
        for i in prange(X.shape[0]):
            s_ = 0.0

            for j in range(X.shape[1]):
                diff = (
                    np.float64(X[i, j])
                    - c[j]
                )
                s_ += diff * diff

            out[i] = s_

def _sqdist(X, c):
    c = np.asarray(
        c,
        dtype=np.float64
    )

    if HAVE_NUMBA:
        out = np.empty(
            X.shape[0],
            dtype=np.float64
        )

        _sqdist_to(
            X,
            c,
            out
        )

        return out

    out = np.empty(
        X.shape[0],
        dtype=np.float64
    )

    for s_ in range(
        0,
        X.shape[0],
        262_144
    ):
        e = min(
            s_ + 262_144,
            X.shape[0]
        )

        out[s_:e] = (
            (
                X[s_:e].astype(np.float64)
                - c
            ) ** 2
        ).sum(axis=1)

    return out

def set_threads(t):
    if HAVE_NUMBA:
        numba.set_num_threads(t)

THREADS_TO_TRY = sorted(
    set(
        THREAD_CANDIDATES
        or [1, N_THREADS]
    )
)

KMEANS = (
    kmeans_numba
    if HAVE_NUMBA
    else kmeans_numpy
)

print(
    "Implementation:",
    IMPLEMENTATION,
    "| thread counts to try:",
    THREADS_TO_TRY if HAVE_NUMBA else "n/a (NumPy)"
)


## 6. Correctness check

We compare each configured D with a direct float64 reference before timing.


In [ ]:
def kmeans_reference(X, C0, iters):
    "Plain float64 Lloyd's algorithm, for checking only."
    X = X.astype(np.float64)
    C = C0.astype(np.float64).copy()
    for _ in range(iters):
        labels = ((X[:, None, :] - C[None]) ** 2).sum(axis=2).argmin(axis=1)
        for k in range(len(C)):
            if (labels == k).any():
                C[k] = X[labels == k].mean(axis=0)
    return C, labels

def correctness_check(n=20_000, d=9, true_k=7, K=17, iters=30):
    rng = np.random.default_rng(1)
    centers = rng.uniform(-100, 1000, (true_k, d))
    X = (centers[rng.integers(0, true_k, n)] + rng.normal(0, 15, (n, d))).astype(DTYPE)
    C0 = X[rng.choice(n, K, replace=False)].copy()
    C_ref, lab_ref = kmeans_reference(X, C0, iters)
    C, lab, *_ = KMEANS(
        X,
        C0,
        iters,
        -1.0,
        -1.0,
        ASSIGNMENT_STABLE_ITERS
    )
    result = {"implementation": IMPLEMENTATION,
              "max |centroid - reference|": float(np.abs(C - C_ref).max()),
              "label agreement %": 100.0 * float((lab == lab_ref).mean())}
    assert result["max |centroid - reference|"] < 1e-2, f"centroids differ from the reference: {result}"
    assert result["label agreement %"] > 99.9, f"labels differ from the reference: {result}"
    return result

for _d in FEATURE_DIMS:
    print(f"CPU correctness D={_d}:", correctness_check(d=_d))


## 7. Benchmark

We record repeated fixed-iteration CPU timings and a separate run to aligned convergence. Final labels are reassigned to the final centroids outside the convergence timer.


In [ ]:
def time_per_iteration(X, C0):
    'Fixed-iteration timing with convergence disabled.'
    for _ in range(WARMUP_RUNS):
        KMEANS(
            X,
            C0,
            min(2, BENCH_ITERS),
            -1.0,
            -1.0,
            ASSIGNMENT_STABLE_ITERS
        )

    t0 = time.perf_counter()

    KMEANS(
        X,
        C0,
        BENCH_ITERS,
        -1.0,
        -1.0,
        ASSIGNMENT_STABLE_ITERS
    )

    est = (
        time.perf_counter() - t0
    ) / BENCH_ITERS

    iters = int(
        min(
            BENCH_MAX_ITERS,
            max(
                BENCH_ITERS,
                np.ceil(
                    BENCH_MIN_SECONDS / est
                )
            )
        )
    )

    times = []

    for _ in range(BENCH_REPEATS):
        t0 = time.perf_counter()

        KMEANS(
            X,
            C0,
            iters,
            -1.0,
            -1.0,
            ASSIGNMENT_STABLE_ITERS
        )

        times.append(
            (
                time.perf_counter() - t0
            ) / iters
        )

    return np.array(times), iters

SUMMARY_PATH = OUT_DIR / "summary.csv"

summary = (
    pd.read_csv(SUMMARY_PATH)
    if RESUME and SUMMARY_PATH.exists()
    else pd.DataFrame()
)

if len(summary):
    stale = summary[
        summary.get(
            "init",
            pd.Series(
                index=summary.index,
                dtype=object
            )
        ) != init_label()
    ]

    if len(stale):
        print(
            f"{len(stale)} earlier rows used different "
            "starting centroids and will be re-run"
        )

    summary = summary.drop(
        stale.index
    )

done = (
    set(summary["file"])
    if len(summary)
    else set()
)

files = find_data_files()

if SIZES is not None:
    files = [
        p for p in files
        if parse_name(p)["rows"] in SIZES
    ]

print(
    f"{len(files)} files, implementation: {IMPLEMENTATION}, "
    f"thread counts to try: "
    f"{THREADS_TO_TRY if HAVE_NUMBA else N_THREADS}"
    + (
        f", {len(done)} already done"
        if done
        else ""
    )
)

cases = [(path, d) for path in files for d in FEATURE_DIMS
         if SWEEP_CASES is None or (parse_name(path)["rows"], parse_name(path)["K"]) in SWEEP_CASES]
print(f"Run plan: {len(cases)} (CSV, D) cases")

with threadpool_limits(N_THREADS):
    for path, feature_dim in cases:
        meta = parse_name(path)
        source_tag = meta["file"]
        run_tag = dimension_tag(source_tag, feature_dim)

        if run_tag in done:
            print(
                f"\n{run_tag}: "
                "already in summary.csv, skipped"
            )
            continue

        t0 = time.perf_counter()
        X, layout = load_dataset(
            path,
            meta,
            feature_dim=feature_dim
        )
        load_s = (
            time.perf_counter() - t0
        )

        n, d = X.shape
        K = meta["K"]

        C0, init_source = initial_centroids(
            meta,
            X,
            layout["feature_columns"]
        )

        init_hash = initial_centroid_sha256(
            C0
        )

        print(
            f"\n{run_tag}: "
            f"X = {X.shape}, K = {K}, "
            f"loaded in {load_s:.1f} s"
        )
        print("  start:", init_source)
        print("  initial centroid SHA256:", init_hash)

        by_threads = {}

        for th in (
            THREADS_TO_TRY
            if HAVE_NUMBA
            else [N_THREADS]
        ):
            set_threads(th)

            t_th, it_th = time_per_iteration(
                X,
                C0
            )

            by_threads[th] = (
                t_th,
                it_th
            )

            print(
                f"  {th} thread(s): "
                f"{np.median(t_th) * 1e3:.3f} ms per iteration "
                f"(min {t_th.min() * 1e3:.3f}, "
                f"max {t_th.max() * 1e3:.3f}; "
                f"{BENCH_REPEATS} runs of {it_th} iterations)"
            )

        best_threads = min(
            by_threads,
            key=lambda k: np.median(
                by_threads[k][0]
            )
        )

        t, bench_iters = (
            by_threads[best_threads]
        )

        med = float(
            np.median(t)
        )

        set_threads(
            best_threads
        )

        print(
            f"  baseline: "
            f"{med * 1e3:.3f} ms per iteration "
            f"with {best_threads} thread(s)"
        )

        t0 = time.perf_counter()

        (
            C,
            labels_at_stop,
            iters,
            shift,
            changed,
            changed_fraction,
            stable_streak,
            convergence_reason,
            history,
        ) = KMEANS(
            X,
            C0,
            MAX_ITER,
            TOL,
            ASSIGNMENT_CHANGE_FRACTION_TOL,
            ASSIGNMENT_STABLE_ITERS,
            True
        )

        total = (
            time.perf_counter() - t0
        )

        converged = (
            convergence_reason
            != "max_iterations"
        )

        pd.DataFrame(
            history
        ).to_csv(
            HISTORY_DIR
            / f"history_{run_tag}.csv",
            index=False
        )

        print(
            f"  run to convergence: "
            f"{iters} iterations, "
            f"{total:.2f} s, "
            f"reason={convergence_reason}, "
            f"changed={changed} "
            f"({changed_fraction * 100:.5f}%), "
            f"shift={shift:.6g}"
        )

        lab, dist = assign_to_centroids(
            X,
            C
        )

        sizes = np.bincount(
            lab,
            minlength=K
        )

        wcss = float(
            np.sum(
                dist.astype(np.float64) ** 2
            )
        )

        if SAVE_ASSIGNMENTS:
            t0 = time.perf_counter()

            out = pd.DataFrame({
                "ID": read_ids(
                    path,
                    layout,
                    n
                ),
                "cluster": lab,
                "distance": dist,
            })

            ASSIGNMENTS_DIR.mkdir(
                parents=True,
                exist_ok=True
            )

            out.to_csv(
                ASSIGNMENTS_DIR
                / f"assignments_{run_tag}.csv",
                index=False
            )

            print(
                "  assignments written in "
                f"{time.perf_counter() - t0:.1f} s"
            )

            del out

        print(
            f"  cluster sizes: "
            f"{sizes.tolist()}"
        )

        final_row = {
            "Implementation": "cpu_numba_v3_aligned"
                if HAVE_NUMBA
                else "cpu_numpy_v3_aligned",
            "Dataset": run_tag,
            "Source dataset": source_tag,
            "Feature columns": json.dumps(layout["feature_columns"]),
            "Rows": n,
            "Features": d,
            "K": K,
            "Execution time": total,
            "Iterations": iters,
            "Per iteration ms":
                total / iters * 1000.0,
            "Converged": bool(converged),
            "Convergence reason":
                convergence_reason,
            "Final assignment changes":
                int(changed),
            "Final assignment change fraction":
                float(changed_fraction),
            "Final assignment change percent":
                float(changed_fraction * 100.0),
            "Stable fraction streak":
                int(stable_streak),
            "Max movement": shift,
            "WCSS": wcss,
            "Cluster sizes":
                json.dumps(sizes.tolist()),
            "Centroids":
                json.dumps(C.astype(float).tolist()),
            "Initial centroid source":
                str(resolve_canonical_initial_path(INIT_DIR, source_tag)),
            "Initial centroid SHA256":
                init_hash,
        }

        pd.DataFrame(
            [final_row]
        ).to_csv(
            OUT_DIR
            / f"final_results_{run_tag}.csv",
            index=False
        )

        row = {
            "file": run_tag,
            "source_file": source_tag,
            "feature_columns": json.dumps(layout["feature_columns"]),
            "rows": n,
            "K": K,
            "features": d,
            "implementation":
                final_row["Implementation"],
            "init": init_label(),
            "canonical_source_file": final_row["Initial centroid source"],
            "initial_centroid_sha256":
                init_hash,
            "threads":
                best_threads
                if HAVE_NUMBA
                else N_THREADS,
            "bench_iters": bench_iters,
            "per_iter_ms_by_threads":
                "; ".join(
                    f"{k}:{np.median(v[0]) * 1e3:.4f}"
                    for k, v in by_threads.items()
                ),
            "per_iter_ms_median":
                med * 1e3,
            "per_iter_ms_min":
                t.min() * 1e3,
            "per_iter_ms_max":
                t.max() * 1e3,
            "ns_per_point_centroid":
                med / (n * K) * 1e9,
            "approx_gflops":
                3 * d * n * K / med / 1e9,
            "conv_iterations": iters,
            "conv_converged":
                bool(converged),
            "conv_reason":
                convergence_reason,
            "conv_total_s": total,
            "conv_final_assignment_changes":
                int(changed),
            "conv_final_assignment_change_fraction":
                float(changed_fraction),
            "conv_final_max_movement":
                shift,
            "empty_clusters":
                int((sizes == 0).sum()),
            "smallest_cluster":
                int(sizes.min()),
            "largest_cluster":
                int(sizes.max()),
            "WCSS":
                wcss,
            "load_s":
                load_s,
        }

        summary = pd.concat(
            [
                summary,
                pd.DataFrame([row])
            ],
            ignore_index=True
        )

        summary.sort_values(
            ["rows", "K", "features"]
        ).to_csv(
            SUMMARY_PATH,
            index=False
        )

        done.add(run_tag)
        del X, lab, dist

summary = (
    summary
    .sort_values(["rows", "K", "features"])
    .reset_index(drop=True)
)

summary.to_csv(
    SUMMARY_PATH,
    index=False
)

summary


## 8. Results

We plot each D/K group separately.


In [ ]:
import matplotlib.pyplot as plt

if len(summary):
    s = summary.sort_values(["rows", "K", "features"])
    fig, ax = plt.subplots(figsize=(10, 6))
    for (d, k), group in s.groupby(["features", "K"]):
        ax.plot(group["rows"], group["per_iter_ms_median"], marker="o", label=f"D={d}, K={k}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Rows")
    ax.set_ylabel("CPU fixed-iteration time (ms)")
    ax.set_title("CPU dimension sweep (warm-up excluded)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "cpu_baseline_time_per_iteration.png", dpi=150)
    plt.show()
    display(s[["file", "rows", "features", "K", "threads", "per_iter_ms_median",
               "conv_iterations", "conv_total_s", "WCSS"]].round(4))
else:
    print("No completed dimension-sweep cases yet.")


## 9. Export

We copy staged assignments first, then package results and subfolders after all timing has finished.


In [ ]:
if SAVE_ASSIGNMENTS and IN_COLAB and ASSIGNMENTS_DIR.exists():
    destination = OUT_DIR / "assignments"
    destination.mkdir(exist_ok=True)
    for path in sorted(ASSIGNMENTS_DIR.glob("*.csv")):
        if path.resolve() != (destination / path.name).resolve():
            shutil.copy2(path, destination / path.name)

zip_path = OUT_DIR / "cpu_baseline_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file() and path.suffix in {".csv", ".json", ".png"}:
            archive.write(path, path.relative_to(OUT_DIR))
print("Saved", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))


## Comparison contract

We use float64, movement tolerance `1e-4`, assignment-change fraction `1e-4` for three consecutive iterations, and a 400-iteration limit. Compare the same N/K/D and verify initialization hashes. CPU post-stop reassignment differs from the GPU export convention; see [results notes](docs/EXPERIMENTS_AND_RESULTS.md).
